In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("init_load_flag", "0")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### Data Reading From Source

In [0]:
df = spark.sql("select * from travel_journal_catalog.silver.google_maps_address_silver")

In [0]:
df.display()

## Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=["id"])


### Dividning New vs Old Records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimGoogleAddressKey, id, create_date, update_date from travel_journal_catalog.gold.DimGoogleAddress''')
    

else:
    df_old = spark.sql('''select 0 DimGoogleAddressKey, 0 id, 0 create_date, 0 update_date from travel_journal_catalog.silver.google_maps_address_silver where 1=0''')

In [0]:
df_old.display()

### Renaming Columns of df_old

In [0]:
df_old = df_old.withColumnRenamed("DimGoogleAddressKey", "old_DimGoogleAddressKey")\
    .withColumnRenamed("id","old_id")\
    .withColumnRenamed("create_date","old_create_date")\
    .withColumnRenamed("update_date","old_update_date")


In [0]:
df_old.display()

## Applying Join with Old Records

In [0]:
from pyspark.sql.functions import col
df_join = df.join(df_old, df.id == df_old.old_id, "left")


In [0]:
df_join.display()


### Separating New vs Old Records

In [0]:
df_new = df_join.filter(df_join.old_DimGoogleAddressKey.isNull())
df_new.display()


In [0]:
df_old = df_join.filter(df_join.old_DimGoogleAddressKey.isNotNull())
df_old.display()

### Prepare df_old

In [0]:
# Dropping all the columns which are not require

df_old = df_old.drop('old_id','old_update_date')

# Renaming "old_create_date column to create_date"
df_old = df_old.withColumnRenamed("old_DimGoogleAddressKey","DimGoogleAddressKey")

df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumnRenamed("old_update_date","update_date")


df_old = df_old.withColumn("create_date",to_timestamp("create_date"))


# Recreating "update_date"
df_old = df_old.withColumn("update_date",current_timestamp())



In [0]:
df_old.display()

### Prepare New df

In [0]:
df_new.display()

In [0]:
# Dropping all the columns which are not require

df_new = df_new.drop('old_DimGoogleAddressKey','old_id','old_update_date','old_create_date')


# Recreating "update_date", "create_date" colums with current timestamp
df_new = df_new.withColumn("update_date",current_timestamp())
df_new = df_new.withColumn("create_date",current_timestamp())



In [0]:
df_new.display()

## Surrogate Key - From 1 

In [0]:
df_new = df_new.withColumn("DimGoogleAddressKey",monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

### Adding Max Surrogatekey

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0

else:
    df_maxsurrogate = spark.sql('''select max(DimGoogleAddressKey) as max_surrogate_key from travel_journal_catalog.gold.DimGoogleAddress''')
    #Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsurrogate.collect()[0]['max_surrogate_key']

In [0]:
print(max_surrogate_key)

In [0]:
df_new = df_new.withColumn("DimGoogleAddressKey", lit(max_surrogate_key)+col("DimGoogleAddressKey"))

In [0]:
df_new.display()

## Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

### SCD Type 1

In [0]:
from delta.tables import DeltaTable

Create Upsert Conditional


In [0]:
if spark.catalog.tableExists("travel_journal_catalog.gold.DimGoogleAddress"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@databricktraveljournal.dfs.core.windows.net/DimGoogleAddress")

    dlt_obj.alias("trg").merge(
        df_final.alias("src"),"trg.DimGoogleAddressKey = src.DimGoogleAddressKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
        
else:
    df_final.write.mode("overwrite")\
    .format("delta")\
    .option("path", "abfss://gold@databricktraveljournal.dfs.core.windows.net/DimGoogleAddress")\
    .saveAsTable("travel_journal_catalog.gold.DimGoogleAddress")

    

In [0]:
%sql

SELECT * FROM travel_journal_catalog.gold.DimGoogleAddress

In [0]:
%sql DESCRIBE CATALOG EXTENDED travel_journal_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS travel_journal_catalog.gold
MANAGED LOCATION 'abfss://gold@databricktraveljournal.dfs.core.windows.net/managed/';